# Q2: Decision Tree from Scratch - BSDS500 Boundary Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat

BASE = r"C:\Users\cqds\Downloads\bsds500archive"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
IMAGES_TRAIN = os.path.join(BASE, "images", "train")
IMAGES_TEST = os.path.join(BASE, "images", "test")
GT_TRAIN = os.path.join(BASE, "ground_truth", "train")
GT_TEST = os.path.join(BASE, "ground_truth", "test")

### Helper functions for loading images and ground truth

In [ ]:
def list_image_files(images_dir, max_images):
    filenames = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    return filenames[:max_images]

def find_gt_path(gt_dir, image_filename):
    stem = os.path.splitext(image_filename)[0]
    candidate = os.path.join(gt_dir, stem + ".mat")
    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"No matching ground-truth .mat file found for image '{image_filename}' "
            f"(expected {candidate})"
        )
    return candidate

def load_bsds_ground_truth(mat_path):
    mat = loadmat(mat_path)
    gt_struct = mat["groundTruth"]
    n_annotators = gt_struct.shape[1]
    boundary_maps = [np.asarray(gt_struct[0, i]["Boundaries"][0, 0], dtype=float)
                      for i in range(n_annotators)]
    consensus = np.mean(boundary_maps, axis=0)
    return (consensus >= 0.5).astype(int)

def get_boundary_labels(gt_array):
    unique_vals = np.unique(gt_array)
    if len(unique_vals) <= 2:
        return (gt_array > 0).astype(int)
    diff_right = gt_array[:, :-1] != gt_array[:, 1:]
    diff_down = gt_array[:-1, :] != gt_array[1:, :]
    boundary = np.zeros_like(gt_array, dtype=bool)
    boundary[:, :-1] |= diff_right
    boundary[:-1, :] |= diff_down
    return boundary.astype(int)

def extract_pixel_features(img_array):
    gray = img_array.mean(axis=2)
    grad_y = np.abs(np.diff(gray, axis=0, prepend=gray[:1, :]))
    grad_x = np.abs(np.diff(gray, axis=1, prepend=gray[:, :1]))
    grad_mag = np.sqrt(grad_x ** 2 + grad_y ** 2)
    feats = np.stack([img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2], gray, grad_mag], axis=-1)
    return feats.reshape(-1, 5)

def load_pixel_dataset(images_dir, gt_dir, max_images, pixels_per_image, seed=0):
    rng = np.random.RandomState(seed)
    filenames = list_image_files(images_dir, max_images)
    X_parts, y_parts = [], []
    total_pixels = 0
    for fname in filenames:
        img = np.array(Image.open(os.path.join(images_dir, fname)).convert("RGB"), dtype=float) / 255.0
        gt_path = find_gt_path(gt_dir, fname)
        gt = load_bsds_ground_truth(gt_path)
        if gt.shape != img.shape[:2]:
            raise ValueError(
                f"Ground truth shape {gt.shape} does not match image shape "
                f"{img.shape[:2]} for '{fname}'"
            )
        labels_full = get_boundary_labels(gt).reshape(-1)
        feats_full = extract_pixel_features(img)
        total_pixels += labels_full.shape[0]
        boundary_idx = np.where(labels_full == 1)[0]
        nonboundary_idx = np.where(labels_full == 0)[0]
        rng.shuffle(boundary_idx)
        rng.shuffle(nonboundary_idx)
        idx = np.concatenate([boundary_idx[:pixels_per_image], nonboundary_idx[:pixels_per_image]])
        X_parts.append(feats_full[idx])
        y_parts.append(labels_full[idx])
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    return X, y, total_pixels, len(filenames)

feature_names = ["R", "G", "B", "gray", "gradient_magnitude"]

### Load training and test pixel samples

In [ ]:
X_train, y_train, total_train_pixels, n_train_images = load_pixel_dataset(
    IMAGES_TRAIN, GT_TRAIN, max_images=40, pixels_per_image=250, seed=0)
X_test, y_test, total_test_pixels, n_test_images = load_pixel_dataset(
    IMAGES_TEST, GT_TEST, max_images=15, pixels_per_image=150, seed=1)

print("Training images used:", n_train_images, " total pixels:", total_train_pixels)
print("Training sample shape:", X_train.shape, " Test sample shape:", X_test.shape)
print("Label balance train:", np.bincount(y_train))

### Gini impurity

In [ ]:
def gini_impurity(labels):
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / len(labels)
    return 1.0 - np.sum(p ** 2)

print("Gini of training set:", round(gini_impurity(y_train), 3))

### Best split search

In [ ]:
def best_split(X, y, max_thresholds=25):
    n_samples, n_features = X.shape
    parent_gini = gini_impurity(y)
    best_gain, best_feature, best_threshold = 0.0, None, None
    for feature in range(n_features):
        values = np.unique(X[:, feature])
        if len(values) > max_thresholds:
            quantiles = np.linspace(0, 1, max_thresholds + 2)[1:-1]
            candidates = np.quantile(values, quantiles)
        else:
            candidates = (values[:-1] + values[1:]) / 2.0
        for t in candidates:
            left_mask = X[:, feature] <= t
            n_left, n_right = left_mask.sum(), (~left_mask).sum()
            if n_left == 0 or n_right == 0:
                continue
            weighted_gini = (n_left / n_samples) * gini_impurity(y[left_mask]) + \
                             (n_right / n_samples) * gini_impurity(y[~left_mask])
            gain = parent_gini - weighted_gini
            if gain > best_gain:
                best_gain, best_feature, best_threshold = gain, feature, t
    return best_feature, best_threshold, best_gain

f, t, g = best_split(X_train, y_train)
print("Best root split -> feature:", feature_names[f], " threshold:", round(t, 4), " gain:", round(g, 4))

### Recursive tree builder

In [ ]:
class Node:
    def __init__(self, prediction=None, feature=None, threshold=None,
                 left=None, right=None, gini=None, n_samples=None):
        self.prediction = prediction
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.gini = gini
        self.n_samples = n_samples

    def is_leaf(self):
        return self.prediction is not None

def majority_class(y):
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]

def build_tree(X, y, depth=0, max_depth=5, min_samples_split=60):
    node_gini = gini_impurity(y)
    if depth >= max_depth or len(y) < min_samples_split or node_gini == 0.0:
        return Node(prediction=majority_class(y), gini=node_gini, n_samples=len(y))
    feature, threshold, gain = best_split(X, y)
    if feature is None or gain == 0.0:
        return Node(prediction=majority_class(y), gini=node_gini, n_samples=len(y))
    left_mask = X[:, feature] <= threshold
    left_child = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples_split)
    right_child = build_tree(X[~left_mask], y[~left_mask], depth + 1, max_depth, min_samples_split)
    return Node(feature=feature, threshold=threshold, left=left_child,
                right=right_child, gini=node_gini, n_samples=len(y))

tree = build_tree(X_train, y_train, max_depth=5, min_samples_split=60)
print("Tree built successfully")

### Predict and evaluate

In [ ]:
def predict_one(node, x):
    while not node.is_leaf():
        node = node.left if x[node.feature] <= node.threshold else node.right
    return node.prediction

def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])

y_pred = predict(tree, X_test)
accuracy = np.mean(y_pred == y_test)
tp = np.sum((y_test == 1) & (y_pred == 1))
tn = np.sum((y_test == 0) & (y_pred == 0))
fp = np.sum((y_test == 0) & (y_pred == 1))
fn = np.sum((y_test == 1) & (y_pred == 0))
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print("Test accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4), " Recall:", round(recall, 4))

### Text structure of the tree

In [ ]:
def print_tree(node, depth=0):
    indent = "    " * depth
    if node.is_leaf():
        label = "boundary" if node.prediction == 1 else "non-boundary"
        print(f"{indent}-> Predict: {label} (samples={node.n_samples}, gini={node.gini:.3f})")
    else:
        fname = feature_names[node.feature]
        print(f"{indent}[{fname} <= {node.threshold:.3f}]  (samples={node.n_samples}, gini={node.gini:.3f})")
        print(f"{indent}True ->")
        print_tree(node.left, depth + 1)
        print(f"{indent}False ->")
        print_tree(node.right, depth + 1)

print_tree(tree)

### Visualize the tree

In [ ]:
def max_depth_of(n):
    if n.is_leaf():
        return 0
    return 1 + max(max_depth_of(n.left), max_depth_of(n.right))

def draw(ax, n, x, y, x_span, depth, depth_total):
    if n.is_leaf():
        label = "boundary" if n.prediction == 1 else "non-boundary"
        text = f"{label}\nn={n.n_samples}"
        box_color = "#cdeccd"
    else:
        text = f"{feature_names[n.feature]}\n<= {n.threshold:.3f}\nn={n.n_samples}"
        box_color = "#cfe0f7"
    ax.text(x, y, text, ha="center", va="center", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.4", fc=box_color, ec="black"))
    if not n.is_leaf():
        child_y = y - 1.0 / (depth_total + 1)
        left_x = x - x_span / 2
        right_x = x + x_span / 2
        ax.plot([x, left_x], [y - 0.03, child_y + 0.03], color="black", lw=1)
        ax.plot([x, right_x], [y - 0.03, child_y + 0.03], color="black", lw=1)
        ax.text((x + left_x) / 2, (y + child_y) / 2, "yes", fontsize=7, color="green")
        ax.text((x + right_x) / 2, (y + child_y) / 2, "no", fontsize=7, color="red")
        draw(ax, n.left, left_x, child_y, x_span / 2, depth + 1, depth_total)
        draw(ax, n.right, right_x, child_y, x_span / 2, depth + 1, depth_total)

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis("off")
depth_total = max_depth_of(tree)
draw(ax, tree, x=0.5, y=0.95, x_span=0.5, depth=0, depth_total=depth_total)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Decision Tree - image boundary detection")
plt.tight_layout()
plt.show()